In [72]:
# Cell 1: Install packages compatible with Colab Enterprise
!pip install --quiet \
    "google-genai>=0.1.1" \
    "google-cloud-aiplatform>=1.38.0" \
    "googlemaps>=4.10.0" \
    "litellm>=1.40.0" \
    "pydantic>=2.0.0"

print("✓ Dependencies installed successfully.")

✓ Dependencies installed successfully.


In [73]:
# Cell 2: Setup Environment and API Keys
import os
import getpass
import google.auth

# Automatically retrieve Project ID inside Cloud Skills Boost / GCP Colab Enterprise
try:
    credentials, project_id = google.auth.default()
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
    print(f"✓ GCP Project ID detected: {project_id}")
except Exception:
    project_id = input("Enter GCP Project ID: ").strip()
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id

# Set region for Vertex AI
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

# Google Maps API Key (provided in Cloud Skills Boost lab credentials or left pane)
if "GOOGLE_MAPS_API_KEY" not in os.environ:
    maps_key = getpass.getpass("Enter Google Maps API Key: ").strip()
    os.environ["GOOGLE_MAPS_API_KEY"] = maps_key

# Optional Third-Party Model API Key (e.g., Anthropic Claude or OpenAI GPT)
if "ANTHROPIC_API_KEY" not in os.environ and "OPENAI_API_KEY" not in os.environ:
    third_party_choice = input("Configure 3rd-party model? (anthropic/openai/skip): ").strip().lower()
    if third_party_choice == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter Anthropic API Key: ").strip()
    elif third_party_choice == "openai":
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ").strip()

✓ GCP Project ID detected: qwiklabs-gcp-00-3e3c5779f847


In [74]:
# Cell 3: Tool 1 - Geocoding via Google Maps API
from typing import Any, Dict, Optional
import googlemaps

def geocode_address(address: str) -> Dict[str, Any]:
    """Converts a textual place name or address into geographic coordinates.

    Uses the Google Maps Geocoding API to resolve a human-readable city, state,
    landmark, or postal address into latitude and longitude coordinates.

    Args:
        address: The place name, city, address, or postal code to geocode
            (e.g., 'Denver, CO', 'Miami, Florida', 'Chicago, IL').

    Returns:
        A dictionary containing:
            - latitude (float): Latitude in decimal degrees.
            - longitude (float): Longitude in decimal degrees.
            - formatted_address (str): Standardized address returned by Google Maps.
            - place_id (str): Unique Google Maps place identifier.
            - status (str): Status string ('OK' or error description).

    Raises:
        ValueError: If the address cannot be resolved or the API returns no results.
    """
    api_key = os.getenv("GOOGLE_MAPS_API_KEY")
    if not api_key:
        return {"status": "ERROR", "message": "GOOGLE_MAPS_API_KEY is not configured."}

    try:
        gmaps = googlemaps.Client(key=api_key)
        geocode_result = gmaps.geocode(address)

        if not geocode_result:
            return {
                "status": "NOT_FOUND",
                "message": f"No coordinates found for address: '{address}'.",
            }

        first_match = geocode_result[0]
        geometry = first_match.get("geometry", {}).get("location", {})

        return {
            "status": "OK",
            "latitude": float(geometry.get("lat")),
            "longitude": float(geometry.get("lng")),
            "formatted_address": first_match.get("formatted_address"),
            "place_id": first_match.get("place_id"),
        }
    except Exception as exc:
        return {
            "status": "ERROR",
            "message": f"Google Maps Geocoding API error: {str(exc)}",
        }

In [75]:
# Cell 4: Tool 2 - National Weather Service (NWS) API
import json
import requests
from typing import Any, Dict, List

def get_nws_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Retrieves real-time weather observations, forecast, and alerts from the NWS.

    Queries the official National Weather Service (api.weather.gov) endpoints by
    first resolving the coordinate points to the local forecast office grid, and
    subsequently querying active alerts and the latest forecast periods.

    Args:
        latitude: Latitude in decimal degrees (e.g., 39.7392).
        longitude: Longitude in decimal degrees (e.g., -104.9903).

    Returns:
        A dictionary containing:
            - status (str): 'OK' or error message.
            - location_meta (dict): Grid ID, forecast office, and radar station.
            - current_forecast (dict): Temperature, wind, short forecast description.
            - active_alerts (list): Active severe weather watches/warnings/advisories.

    Raises:
        requests.RequestException: If network connectivity or NWS API fails.
    """
    headers = {
        "User-Agent": "(CloudSkillsBoostWeatherAgent/2.0, contact@cloudskillsboost.google)",
        "Accept": "application/geo+json",
    }

    try:
        # Step 1: Query /points endpoint to obtain grid and forecast URL
        points_url = f"https://api.weather.gov/points/{latitude:.4f},{longitude:.4f}"
        point_resp = requests.get(points_url, headers=headers, timeout=10)

        if point_resp.status_code != 200:
            return {
                "status": "ERROR",
                "message": f"NWS points lookup failed with status code {point_resp.status_code}.",
            }

        point_data = point_resp.json()
        props = point_data.get("properties", {})
        forecast_url = props.get("forecast")
        grid_id = props.get("gridId")
        radar_station = props.get("radarStation")

        # Step 2: Query the forecast endpoint for current conditions & outlook
        forecast_summary = {}
        if forecast_url:
            fc_resp = requests.get(forecast_url, headers=headers, timeout=10)
            if fc_resp.status_code == 200:
                fc_periods = fc_resp.json().get("properties", {}).get("periods", [])
                if fc_periods:
                    current_period = fc_periods[0]
                    forecast_summary = {
                        "period_name": current_period.get("name"),
                        "temperature": current_period.get("temperature"),
                        "temperature_unit": current_period.get("temperatureUnit"),
                        "wind_speed": current_period.get("windSpeed"),
                        "wind_direction": current_period.get("windDirection"),
                        "short_forecast": current_period.get("shortForecast"),
                        "detailed_forecast": current_period.get("detailedForecast"),
                    }

        # Step 3: Check for active alerts at coordinates
        alerts_url = f"https://api.weather.gov/alerts/active?point={latitude:.4f},{longitude:.4f}"
        alerts_resp = requests.get(alerts_url, headers=headers, timeout=10)
        active_alerts: List[Dict[str, str]] = []

        if alerts_resp.status_code == 200:
            alert_features = alerts_resp.json().get("features", [])
            for feat in alert_features:
                alert_props = feat.get("properties", {})
                active_alerts.append({
                    "event": alert_props.get("event"),
                    "severity": alert_props.get("severity"),
                    "urgency": alert_props.get("urgency"),
                    "headline": alert_props.get("headline"),
                    "instruction": alert_props.get("instruction") or "Follow local emergency guidance."
                })

        return {
            "status": "OK",
            "grid_id": grid_id,
            "radar_station": radar_station,
            "forecast": forecast_summary,
            "active_alerts_count": len(active_alerts),
            "active_alerts": active_alerts,
        }

    except Exception as exc:
        return {
            "status": "ERROR",
            "message": f"Failed to retrieve NWS weather data: {str(exc)}",
        }

In [76]:
# Cell 5: Agent Implementation with Vertex AI and 3rd-Party Model Support
import json
import logging
import warnings
from typing import Optional
from google import genai
from google.genai import types
import litellm

# Suppress known asyncio task cleanup warning on Python 3.12 / Colab
warnings.filterwarnings("ignore", category=ResourceWarning)
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

WEATHER_AGENT_INSTRUCTIONS = """
You are a Weather Intelligence Agent deployed on the Google Cloud Agent Platform.
Your purpose is to deliver accurate weather reports, forecasts, and safety advisories for any user-requested location in the USA.

Follow this execution workflow:
1. When a user mentions a city, state, or address, call `geocode_address` with the user's location to get latitude and longitude.
2. Use the resulting latitude and longitude to call `get_nws_weather` from the National Weather Service.
3. Synthesize the findings into a clear, structured report with:
   - 📍 **Location**: Formatted address and station info.
   - 🌡️ **Current Conditions**: Temperature, wind, and sky condition.
   - ⚠️ **Alerts & Warnings**: Detail any NWS alerts (Watch, Warning, Advisory). If none, state 'No active alerts'.
   - 💡 **Actionable Guidance**: Advice for travel, outdoor activities, or severe weather precautions.
"""

class MultiModelWeatherAgent:
    """Agent that runs either on Vertex AI Gemini or 3rd-Party LLMs (Claude/GPT)."""

    def __init__(self, provider: str = "gemini", model_name: Optional[str] = None):
        """
        Args:
            provider: 'gemini' for Google GenAI on Vertex AI, or 'third_party' (Claude, GPT via LiteLLM).
            model_name: Name of the model. Defaults to 'gemini-2.5-flash' or 'gpt-4o'.
        """
        self.provider = provider.lower()
        self.tools = [geocode_address, get_nws_weather]
        self.tool_map = {
            "geocode_address": geocode_address,
            "get_nws_weather": get_nws_weather,
        }

        if self.provider == "gemini":
            self.model_name = model_name or "gemini-2.5-flash"
            project = os.getenv("GOOGLE_CLOUD_PROJECT")
            location = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
            api_key = os.getenv("GEMINI_API_KEY")

            # In Cloud Skills Boost Colab Enterprise, use Vertex AI authentication
            if project and not api_key:
                self.client = genai.Client(vertexai=True, project=project, location=location)
            else:
                self.client = genai.Client(api_key=api_key)
        else:
            self.model_name = model_name or os.getenv("THIRD_PARTY_MODEL", "gpt-4o")

    def run(self, user_prompt: str) -> str:
        """Executes the agent workflow against the configured model."""
        if self.provider == "gemini":
            return self._run_gemini(user_prompt)
        else:
            return self._run_third_party(user_prompt)

    def _run_gemini(self, user_prompt: str) -> str:
        """Gemini tool-calling execution using google-genai SDK."""
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=WEATHER_AGENT_INSTRUCTIONS,
                tools=self.tools,
                temperature=0.2,
            ),
        )
        return response.text

    def _run_third_party(self, user_prompt: str) -> str:
        """Third-party model tool execution (Claude/GPT) via LiteLLM."""
        openai_tools = [
            {
                "type": "function",
                "function": {
                    "name": "geocode_address",
                    "description": "Converts a place name to latitude and longitude via Google Maps.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "address": {"type": "string", "description": "City, state or address to geocode."}
                        },
                        "required": ["address"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "get_nws_weather",
                    "description": "Gets weather forecast and active alerts from National Weather Service.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "latitude": {"type": "number", "description": "Latitude coordinate."},
                            "longitude": {"type": "number", "description": "Longitude coordinate."},
                        },
                        "required": ["latitude", "longitude"],
                    },
                },
            },
        ]

        messages = [
            {"role": "system", "content": WEATHER_AGENT_INSTRUCTIONS},
            {"role": "user", "content": user_prompt},
        ]

        for _ in range(5):
            response = litellm.completion(
                model=self.model_name,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                return msg.content

            for tool_call in msg.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                tool_fn = self.tool_map.get(fn_name)
                tool_result = tool_fn(**fn_args) if tool_fn else {"error": "Tool not found"}

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result),
                })

        return messages[-1].content

In [77]:
# Cell 6: Multi-City Test Code
import time
import pandas as pd
from IPython.display import display, Markdown

TEST_LOCATIONS = [
    {"name": "Miami, FL", "climate": "Subtropical / Hurricane Watch Zone"},
    {"name": "Denver, CO", "climate": "High Altitude / Mountain Snow"},
    {"name": "Chicago, IL", "climate": "Midwest / Lake Effect & Wind"},
    {"name": "Phoenix, AZ", "climate": "Arid Desert / Extreme Heat"},
    {"name": "Seattle, WA", "climate": "Pacific Northwest / Rain"},
]

def run_multicity_tests(agent: MultiModelWeatherAgent):
    """Executes verification tests across all designated US cities."""
    print("=" * 75)
    print(f"RUNNING TEST SUITE: Model Provider = {agent.provider.upper()} ({agent.model_name})")
    print("=" * 75)

    summary_records = []

    for item in TEST_LOCATIONS:
        loc = item["name"]
        climate = item["climate"]
        query = f"Check current weather and any active weather alerts for {loc}. Provide a summary and recommendations."
        print(f"\nTesting: {loc} ({climate})...")

        start_time = time.time()
        try:
            output = agent.run(query)
            elapsed = round(time.time() - start_time, 2)

            summary_records.append({
                "Location": loc,
                "Climate Category": climate,
                "Status": "PASS",
                "Latency (s)": elapsed,
            })

            display(Markdown(f"### Results for {loc}"))
            display(Markdown(output))
            print("-" * 75)

        except Exception as e:
            elapsed = round(time.time() - start_time, 2)
            summary_records.append({
                "Location": loc,
                "Climate Category": climate,
                "Status": "FAIL",
                "Latency (s)": elapsed,
            })
            print(f"FAILED for {loc}: {str(e)}")

    print("\n" + "=" * 75)
    print("TEST EXECUTION SCORECARD")
    print("=" * 75)
    df_summary = pd.DataFrame(summary_records)
    print(df_summary.to_string(index=False))

In [78]:
# Cell 7: Execute with Gemini (Default)
gemini_agent = MultiModelWeatherAgent(provider="gemini", model_name="gemini-2.5-flash")
run_multicity_tests(gemini_agent)

RUNNING TEST SUITE: Model Provider = GEMINI (gemini-2.5-flash)

Testing: Miami, FL (Subtropical / Hurricane Watch Zone)...


### Results for Miami, FL

I'm sorry, I wasn't able to retrieve the weather information because of an issue with the geocoding service. It seems there's an invalid API key. Please try again later.

---------------------------------------------------------------------------

Testing: Denver, CO (High Altitude / Mountain Snow)...


### Results for Denver, CO

...

I'm sorry, but I was unable to retrieve the weather information. There was an issue with the Geocoding API key, which is necessary to convert "Phoenix, AZ" into geographic coordinates. Please try again later or contact support if the problem persists.

---------------------------------------------------------------------------

Testing: Seattle, WA (Pacific Northwest / Rain)...


### Results for Seattle, WA

I'm sorry, I wasn't able to retrieve the weather information due to an issue with the geocoding service. It seems there's an invalid API key. Please try again later.

---------------------------------------------------------------------------

TEST EXECUTION SCORECARD
   Location                   Climate Category Status  Latency (s)
  Miami, FL Subtropical / Hurricane Watch Zone   PASS         2.02
 Denver, CO      High Altitude / Mountain Snow   PASS         2.43
Chicago, IL       Midwest / Lake Effect & Wind   PASS         3.48
Phoenix, AZ         Arid Desert / Extreme Heat   PASS         2.84
Seattle, WA           Pacific Northwest / Rain   PASS         3.35


In [79]:
# Cell 8: Execute with Claude or GPT-4o
# Set your model name (e.g., 'claude-3-5-sonnet-20241022' or 'gpt-4o')
if os.getenv("ANTHROPIC_API_KEY"):
    third_party_agent = MultiModelWeatherAgent(
        provider="third_party",
        model_name="claude-sonnet-5"
    )
    run_multicity_tests(third_party_agent)
elif os.getenv("OPENAI_API_KEY"):
    third_party_agent = MultiModelWeatherAgent(
        provider="third_party",
        model_name="gpt-4o"
    )
    run_multicity_tests(third_party_agent)
else:
    print("Skipping 3rd-party test: Neither ANTHROPIC_API_KEY nor OPENAI_API_KEY is configured.")

RUNNING TEST SUITE: Model Provider = THIRD_PARTY (claude-sonnet-5)

Testing: Miami, FL (Subtropical / Hurricane Watch Zone)...


### Results for Miami, FL

## 🌤️ Weather Report: Miami, FL

📍 **Location**: Miami, FL (approx. 25.76°N, -80.19°W)
- **NWS Office**: Miami, FL (Grid: MFL)
- **Radar Station**: KAMX

*Note: Location was resolved using standard public coordinates for Miami, as the geocoding service experienced a temporary API key error. Coordinates should be accurate for downtown/central Miami.*

---

🌡️ **Current Conditions (Today)**
- **Temperature**: 88°F (Heat index up to 100°F)
- **Wind**: 8 mph, from the North
- **Sky**: Mostly sunny, with a chance of showers and thunderstorms developing after 1 PM
- **Precipitation Chance**: 50% (light rainfall expected, less than 0.1")

---

⚠️ **Alerts & Warnings** — 1 Active Alert

**🌊 Rip Current Statement** (Moderate Severity)
- **Valid**: Sept 24, 11:54 AM EDT → Sept 26, 8:00 AM EDT
- **Instructions**: Swim only near a lifeguard. If caught in a rip current: **stay calm and float** — do not fight the current. If possible, swim parallel to the shore to escape it. If you can't escape, face the shore and signal/call for help.

---

💡 **Actionable Guidance**

🏖️ **Beach & Water Activities**: Exercise caution — rip currents are a serious hazard through Friday morning. Only swim in areas with lifeguards on duty, and keep a close eye on children and inexperienced swimmers.

🌞 **Heat Safety**: With a heat index near 100°F, stay hydrated, wear light clothing, and limit prolonged sun exposure, especially midday.

⛈️ **Afternoon Plans**: Thunderstorms are likely after 1 PM. If you're heading outdoors (parks, boating, outdoor dining), plan for morning activities and have an indoor backup for the afternoon. Lightning safety applies if storms roll in.

🚗 **Travel**: No major travel disruptions expected, but afternoon showers could cause brief, localized street flooding — allow extra time if driving during storms.

Let me know if you'd like a multi-day forecast or details for a specific Miami neighborhood!

---------------------------------------------------------------------------

Testing: Denver, CO (High Altitude / Mountain Snow)...


### Results for Denver, CO

...

# 🌤️ Weather Report: Phoenix, Arizona

📍 **Location**
- Phoenix, AZ (33.4484° N, 112.0740° W)
- NWS Forecast Office: PSR (Phoenix)
- Radar Station: KIWA

🌡️ **Current Conditions (Today)**
- **Temperature**: 97°F
- **Wind**: 5 mph from the ENE (light breeze)
- **Sky**: Mostly sunny, with a slight chance of showers and thunderstorms developing after 4 PM
- **Precipitation Chance**: 20% (less than 0.1" expected if any falls)

⚠️ **Alerts & Warnings**
- **No active alerts** for the Phoenix area at this time.

💡 **Actionable Guidance**
- **Heat Safety**: 97°F is hot — stay hydrated, wear sunscreen, and avoid prolonged direct sun exposure between 11 AM–4 PM when UV/heat intensity peaks.
- **Outdoor Activities**: Morning hours are ideal for outdoor exercise or activities before temperatures climb. If planning anything after 4 PM, keep an eye on the sky for pop-up thunderstorms.
- **Travel**: Roads should be clear and dry for most of the day. If driving during the evening, watch for sudden localized rain or dust from any thunderstorm activity — visibility can drop quickly in monsoon-season storms.
- **General Tip**: Since it's a "slight chance" (20%) of storms, no need to cancel outdoor plans, but have an indoor backup for late afternoon/evening.

Let me know if you'd like an extended forecast or a check on nearby areas!

---------------------------------------------------------------------------

Testing: Seattle, WA (Pacific Northwest / Rain)...


### Results for Seattle, WA

## 🌦️ Weather Report for Seattle, WA

📍 **Location**: Seattle, Washington (approx. 47.6062° N, 122.3321° W)
- NWS Forecast Office: Seattle (Grid ID: SEW)
- Radar Station: KATX

*Note: I was unable to verify this location via the geocoding service due to an API key error, so coordinates are based on Seattle's standard public location. Let me know if you meant a specific address/neighborhood, and I can refine this once geocoding is restored.*

🌡️ **Current Conditions (Today)**
- **Temperature**: 62°F, trending down to around 60°F in the afternoon
- **Wind**: Light, ~6 mph from the North
- **Sky**: Mostly cloudy with a slight chance of light rain after 2 PM (20% chance of precipitation, less than 0.1" expected)

⚠️ **Alerts & Warnings**
- **No active alerts** for the Seattle area at this time. ✅

💡 **Actionable Guidance**
- **Outdoor Activities**: Good conditions overall — mild temps and only a slight rain chance. A light jacket is sufficient; consider a compact umbrella just in case for after 2 PM.
- **Travel**: No weather-related travel disruptions expected. Roads should remain safe and dry for most of the day.
- **General Precautions**: No severe weather precautions needed today. Typical Pacific Northwest overcast conditions — nothing out of the ordinary.

Let me know if you'd like an extended forecast or a check on a more specific Seattle neighborhood!

---------------------------------------------------------------------------

TEST EXECUTION SCORECARD
   Location                   Climate Category Status  Latency (s)
  Miami, FL Subtropical / Hurricane Watch Zone   PASS        16.69
 Denver, CO      High Altitude / Mountain Snow   PASS        13.19
Chicago, IL       Midwest / Lake Effect & Wind   PASS        12.35
Phoenix, AZ         Arid Desert / Extreme Heat   PASS        12.43
Seattle, WA           Pacific Northwest / Rain   PASS        15.11
